# Gas chromatography: Agilent OpenLab EZChrom

`GCData` reads a `.rslt` result folder directly, without OpenLab: the
chromatogram of every detector and the vendor's analysis (peak table and
amounts per species). The example is the test fixture under
`tests/chromatography/agilent/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lab_data_parser as ldp

In [ ]:
RSLT = next((Path("..") / "tests" / "chromatography" / "agilent" / "ezchrom-test-data").glob("*.rslt"))
gc_raw = ldp.GCData.from_ezchrom_rslt(RSLT)
gc_raw

The `injection` dimension is always present (length 1 here), and every
detector has its own retention time axis.

In [ ]:
print("detectors:", gc_raw.detectors)
print("injections:", gc_raw.n_injections)
print("analysis:", gc_raw.ds.attrs.get("analysis_source"))

## Chromatograms

In [ ]:
fig, axes = plt.subplots(len(gc_raw.detectors), 1, figsize=(7, 5), sharex=True)
for ax, detector in zip(axes, gc_raw.detectors):
    chromatogram = gc_raw.get_chromatogram(detector, injection=0)
    ax.plot(gc_raw.get_retention_time(detector) / 60, chromatogram)
    ax.set_ylabel(f"{detector} / {gc_raw.get_unit(detector)}")
axes[-1].set_xlabel("retention time / min")
plt.show()

## Peak table and amounts

`get_peaks` returns the integrated peaks as a `pandas.DataFrame`, `get_amounts`
the amounts per species and detector. The species names are those of the GC
method; `rename_species` maps them onto the standard convention.

In [ ]:
gc_raw.get_peaks()[["detector", "species", "retention_time", "height", "area"]]

In [ ]:
gc = gc_raw.rename_species()
gc.get_amounts(detector="tcd")